In [ ]:
# ==========================================================
# FEATURE ENGINEERING NOTEBOOK
# ==========================================================
#  OBJECTIVE:
# - Clean data
# - Create meaningful features
# - Prepare dataset for modeling

In [14]:
# ==========================================================
#  STEP 1 — LOAD LIBRARIES
# ==========================================================
import pandas as pd
import numpy as np

In [15]:
# ==========================================================
#  STEP 2 — LOAD DATA
# ==========================================================
df = pd.read_csv("../data/train.csv")

df.head()

,customer_id,age,gender,country,tenure_months,monthly_spend,num_transactions,last_login_days,support_tickets,payment_method,is_active,total_spend,avg_transaction_value,target_churn,target_revenue,target_fraud
0,1502,59,Male,Germany,2,185.360890,52,18,5,PayPal,0,370.721781,6.994751,1,382.034265,0
1,2587,25,Female,US,54,108.879222,35,17,7,Card,1,5879.477977,163.318833,0,217.996705,1
2,2654,38,Female,Germany,48,221.927474,5,2,5,PayPal,1,10652.518741,1775.419790,0,208.955628,0
3,1056,48,Male,US,26,275.346433,14,26,5,Card,1,7159.007247,477.267150,0,248.913546,0
4,706,53,Male,UK,15,444.016783,95,20,0,Card,1,6660.251746,69.377622,0,768.714022,0


In [16]:
# ==========================================================
# STEP 3 — BASIC CLEANING (PRE-BUILT)
# ==========================================================
#  Fill missing values (simple approach)

df.fillna(0, inplace=True)

# Convert categorical variables
df = pd.get_dummies(df, drop_first=True)

print("After Encoding Shape:", df.shape)

After Encoding Shape: (4000, 19)


In [17]:
# ==========================================================
# TARGET VALIDATION 
# ==========================================================
print("\nTarget Distribution:")
print(df["target_churn"].value_counts(normalize=True))


Target Distribution:
target_churn
0    0.86925
1    0.13075
Name: proportion, dtype: float64


In [18]:
# ==========================================================
# EXISTING FEATURE CREATION (EXAMPLE)
# ==========================================================
#  Example feature already created for you

df["spend_per_transaction"] = df["total_spend"] / (df["num_transactions"] + 1)

df.head()

,customer_id,age,tenure_months,monthly_spend,num_transactions,last_login_days,support_tickets,is_active,total_spend,avg_transaction_value,target_churn,target_revenue,target_fraud,gender_Male,country_India,country_UK,country_US,payment_method_Crypto,payment_method_PayPal,spend_per_transaction
0,1502,59,2,185.360890,52,18,5,0,370.721781,6.994751,1,382.034265,0,True,False,False,False,False,True,6.994751
1,2587,25,54,108.879222,35,17,7,1,5879.477977,163.318833,0,217.996705,1,False,False,False,True,False,False,163.318833
2,2654,38,48,221.927474,5,2,5,1,10652.518741,1775.419790,0,208.955628,0,False,False,False,False,False,True,1775.419790
3,1056,48,26,275.346433,14,26,5,1,7159.007247,477.267150,0,248.913546,0,True,False,False,True,False,False,477.267150
4,706,53,15,444.016783,95,20,0,1,6660.251746,69.377622,0,768.714022,0,True,False,True,False,False,False,69.377622


In [19]:
# ==========================================================
#  IMPORTANT — CHOOSE YOUR TARGET
# ==========================================================
#  STUDENT TASK:
# Choose ONE target column based on your problem

# Options:
# - target_churn
# - target_fraud
# - target_revenue

TARGET_COLUMN = "target_churn"   #  CHANGE THIS

In [20]:
# ==========================================================
#  STUDENT TASK 1 — CREATE NEW FEATURES
# ==========================================================
#  Goal: Create at least 2 meaningful features

# ============================================================================
# FEATURE ENGINEERING FOR CHURN PREDICTION
# ============================================================================
 
# 1. SPEND VELOCITY - How much they spend per month on average
# Higher values indicate more valuable customers, lower values may indicate declining interest
df['spend_velocity'] = df['total_spend'] / df['tenure_months']
 
# 2. TRANSACTION FREQUENCY - Transactions per month
# Higher frequency = more engaged, lower = potential churn risk
df['transaction_frequency'] = df['num_transactions'] / df['tenure_months']
 
# 3. ENGAGEMENT SCORE - Composite metric of customer engagement
# Considers activity status, recent login, and support needs
# Lower scores indicate disengagement (churn risk)
df['engagement_score'] = (
    (df['is_active'] * 40) +  # Active status worth 40 points
    (np.maximum(30 - df['last_login_days'], 0)) +  # Recent login worth up to 30 points
    (np.maximum(20 - df['support_tickets'] * 2, 0))  # Fewer tickets = better (max 20 points)
)
 
# 4. SUPPORT INTENSITY - Support tickets per month
# High values may indicate dissatisfaction
df['support_intensity'] = df['support_tickets'] / df['tenure_months']
 
# 5. INACTIVITY RATIO - How long since login relative to tenure
# Higher values indicate growing disengagement
df['inactivity_ratio'] = df['last_login_days'] / df['tenure_months']
 
# 6. IS_HIGH_VALUE - Binary indicator for high-value customers
# Based on total spend above 75th percentile
spend_threshold = df['total_spend'].quantile(0.75)
df['is_high_value'] = (df['total_spend'] >= spend_threshold).astype(int)
 
# 7. TRANSACTION VALUE CONSISTENCY - Ratio of avg transaction to monthly spend
# Helps identify if customers are making consistent purchases
df['transaction_consistency'] = df['avg_transaction_value'] / df['monthly_spend']
 
# 8. LOYALTY INDICATOR - Combination of tenure and activity
# Long tenure + active = loyal customer
df['loyalty_score'] = df['tenure_months'] * (df['is_active'] + 0.5)
 
# 9. CHURN RISK SCORE - Composite risk indicator
# Higher scores = higher churn risk
df['churn_risk_score'] = (
    (df['is_active'] == 0) * 30 +  # Not active
    (df['last_login_days'] > 14) * 20 +  # Haven't logged in recently
    (df['support_tickets'] > 5) * 15 +  # High support needs
    (df['tenure_months'] < 6) * 25 +  # New customers at risk
    (df['transaction_frequency'] < 1) * 10  # Low transaction frequency
)
 
# 10. SPENDING TREND INDICATOR - Comparing monthly spend to average transaction value
# Can indicate if recent behavior differs from historical patterns
df['spend_pattern_ratio'] = df['monthly_spend'] / (df['avg_transaction_value'] + 1)
 
print("ENGINEERED FEATURES:\n")
 
# Display new features
new_features = [
    'spend_velocity', 'transaction_frequency', 'engagement_score',
    'support_intensity', 'inactivity_ratio', 'is_high_value',
    'transaction_consistency', 'loyalty_score', 'churn_risk_score',
    'spend_pattern_ratio'
]
 
print(df[['customer_id', 'target_churn'] + new_features])
 
print("\n" + "="*80 + "\n")

ENGINEERED FEATURES:

      customer_id  target_churn  spend_velocity  transaction_frequency  \
0            1502             1      185.360890              26.000000   
1            2587             0      108.879222               0.648148   
2            2654             0      221.927474               0.104167   
3            1056             0      275.346433               0.538462   
4             706             0      444.016783               6.333333   
...           ...           ...             ...                    ...   
3995         3336             1      171.334646               0.666667   
3996         1921             0       60.644555               0.800000   
3997         3716             0      257.557570               2.411765   
3998         4647             1      169.479562               0.129032   
3999          947             0       40.535969               0.735849   

      engagement_score  support_intensity  inactivity_ratio  is_high_value  \
0          

In [21]:
# ==========================================================
# FEATURE VALIDATION 
# ==========================================================

print("\nFeature Correlation with Target:")
print(df.corr()[TARGET_COLUMN].sort_values(ascending=False).head(10))


Feature Correlation with Target:
target_churn               1.000000
inactivity_ratio           0.204869
transaction_consistency    0.146720
avg_transaction_value      0.125150
spend_per_transaction      0.125150
transaction_frequency      0.103628
spend_pattern_ratio        0.094917
support_intensity          0.093953
churn_risk_score           0.090994
last_login_days            0.061782
Name: target_churn, dtype: float64


In [22]:
# ==========================================================
# FEATURE SELECTION
# ==========================================================

# Remove ID column
df.drop(columns=["customer_id"], inplace=True, errors="ignore")

# OPTIONAL: Remove leakage features if detected
# Example:
# df.drop(columns=["leakage_column"], inplace=True)

In [23]:
# ==========================================================
# FINAL DATA PREPARATION
# ==========================================================

features = [
    "spend_velocity",
    "transaction_frequency",
    "engagement_score",
    "support_intensity",
    "inactivity_ratio",
    "is_high_value",
    "transaction_consistency",
    "loyalty_score",
    "churn_risk_score",
    "spend_pattern_ratio"
]

X = df[features]
y = df[TARGET_COLUMN]


print("Final Feature Shape:", X.shape)

Final Feature Shape: (4000, 10)


In [24]:
# ==========================================================
#  STEP 6 — SAVE PROCESSED DATA
# ==========================================================
# This will be used in model training notebook

processed_df = pd.concat([X, y], axis=1)

processed_df.to_csv("../data/processed_train.csv", index=False)

print(" Processed data saved successfully!")

 Processed data saved successfully!


In [25]:
# ==========================================================
#  FINAL INSTRUCTIONS
# ==========================================================
# WHAT YOU MUST DO:
# ✔ Choose correct TARGET_COLUMN
# ✔ Create at least 2 new features
# ✔ Ensure feature logic makes sense

# WHAT YOU SHOULD THINK ABOUT:
#  Which features help predict your target?
#  Are you using meaningful combinations?
#  Can you improve model performance with better features?

#  COMMON MISTAKE:
# If your model performs badly → feature engineering is weak